In [ ]:
import os
import h5py
import torch
from torch.utils.data import Dataset
import numpy as np
import torch
import inspect
from torch import optim
import torch.nn as nn
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from numpy import random
import cv2
from numpy import identity

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
print(os.listdir('/content/drive/MyDrive'))

In [ ]:
!unzip -q "/content/drive/MyDrive/landslide4sense.zip" -d /content/

In [ ]:
!rm -rf "/content/landslide4sense"

In [ ]:
print(os.listdir('/content/landslide4sense/'))

In [ ]:
import os
import inspect
import h5py
import numpy as np
import cv2
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset


def _affine_border_kwarg():
    """albumentations renamed Affine's border-mode argument from 'mode' (older
    versions) to 'border_mode' (2.0+). Passing the wrong one is silently
    ignored with a warning — no error, but reflect-padding quietly stops
    happening at rotated/scaled edges. This picks whichever name the
    installed version actually accepts."""
    params = inspect.signature(A.Affine.__init__).parameters
    return "border_mode" if "border_mode" in params else "mode"


def compute_topographical_features(dem, slope, res=10.0):
    """Kept for later — not currently used since these need special handling
    under geometric augmentation (see note in the chat response)."""
    dem_padded = np.pad(dem, pad_width=1, mode="edge")
    dy, dx = np.gradient(dem_padded, res)
    d2y, _ = np.gradient(dy, res)
    _, d2x = np.gradient(dx, res)

    dx = dx[1:-1, 1:-1]
    dy = dy[1:-1, 1:-1]
    d2x = d2x[1:-1, 1:-1]
    d2y = d2y[1:-1, 1:-1]

    aspect = np.arctan2(-dy, dx)
    northness = np.cos(aspect)
    eastness = np.sin(aspect)
    curvature = d2x + d2y
    return northness, eastness, curvature


def compute_normalization(img_dir, file_ids):
    """Unchanged from your version."""
    N_CHANNELS = 16
    channel_sum = np.zeros(N_CHANNELS, dtype=np.float64)
    channel_squared_sum = np.zeros(N_CHANNELS, dtype=np.float64)
    pixel_count = 0
    eps = 1e-6

    for file_id in file_ids:
        img_path = os.path.join(img_dir, f"image_{file_id}.h5")
        if not os.path.exists(img_path):
            continue

        with h5py.File(img_path, "r") as f:
            raw_image = f["img"][:]

        blue = raw_image[:, :, 1].astype(np.float32)
        green = raw_image[:, :, 2].astype(np.float32)
        red = raw_image[:, :, 3].astype(np.float32)
        b5 = raw_image[:, :, 4].astype(np.float32)
        b6 = raw_image[:, :, 5].astype(np.float32)
        b7 = raw_image[:, :, 6].astype(np.float32)
        nir = raw_image[:, :, 7].astype(np.float32)
        swir1 = raw_image[:, :, 10].astype(np.float32)
        swir2 = raw_image[:, :, 11].astype(np.float32)
        slope = raw_image[:, :, 12].astype(np.float32)
        dem   = raw_image[:, :, 13].astype(np.float32)
        
        _, _, curvature = compute_topographical_features(dem, slope)
        
        ndvi = (nir - red) / (nir + red + eps)
        bsi = ((swir1 + red) - (nir + blue)) / ((swir1 + red) + (nir + blue) + eps)
        ndwi = (green - nir) / (green + nir + eps)
        ndmi = (nir - swir1) / (nir + swir1 + eps)
        nbr = (nir - swir2) / (nir + swir2 + eps)

        image_17ch = np.stack(
            [dem, slope, curvature, blue, green, red, nir, b5, b6, b7, swir1, swir2, ndvi, bsi, ndwi, ndmi, nbr], axis=-1
        )  # final 17 channel raster
        
        image_17ch = np.nan_to_num(image_17ch, nan=0.0)  # Replace NaN values with 0.0
        
        h, w, _ = image_17ch.shape
        
        channel_sum += np.sum(image_17ch, axis=(0, 1))
        channel_squared_sum += np.sum(image_17ch ** 2, axis=(0, 1))
        pixel_count += h * w

    means = channel_sum / pixel_count
    stds = np.sqrt((channel_squared_sum / pixel_count) - (means ** 2))
    return means.astype(np.float32), stds.astype(np.float32)


def compute_sample_weights(mask_dir, file_ids):
    """Per-sample weight for WeightedRandomSampler: patches containing at
    least one landslide pixel get a higher weight than pure-background
    patches, sized by inverse frequency so on average every batch has a
    reasonable mix of both, instead of leaving it to chance.
    """
    positive_flags = []
    for file_id in file_ids:
        mask_path = os.path.join(mask_dir, f"mask_{file_id}.h5")
        with h5py.File(mask_path, "r") as f:
            mask = f["mask"][:]
        positive_flags.append(1 if mask.sum() > 0 else 0)

    positive_flags = np.array(positive_flags)
    n_pos = int(positive_flags.sum())
    n_neg = int(len(positive_flags) - n_pos)

    weight_pos = len(positive_flags) / (2.0 * n_pos) if n_pos > 0 else 0.0
    weight_neg = len(positive_flags) / (2.0 * n_neg) if n_neg > 0 else 0.0

    sample_weights = np.where(positive_flags == 1, weight_pos, weight_neg)
    return sample_weights.astype(np.float32), n_pos, n_neg


def train_transform(means, stds):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(
            translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
            scale=(0.9, 1.1),
            rotate=(-10, 10),
            p=0.5,
            **{_affine_border_kwarg(): cv2.BORDER_REFLECT},
        ),
        A.Normalize(mean=list(means), std=list(stds), max_pixel_value=1.0),
        ToTensorV2(),
    ])


def val_transform(means, stds):
    return A.Compose([
        A.Normalize(mean=list(means), std=list(stds), max_pixel_value=1.0),
        ToTensorV2(),
    ])


class LandslideDataset(Dataset):
    def __init__(self, img_dir, mask_dir=None, transform=None, file_ids=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform

        if file_ids is not None:
            self.file_ids = file_ids
        else:
            self.file_ids = sorted(
                int(f.split("_")[1].split(".")[0])
                for f in os.listdir(img_dir)
                if f.endswith(".h5")
            )

    def __len__(self):
        return len(self.file_ids)

    def __getitem__(self, idx):
        file_id = self.file_ids[idx]
        img_name = f"image_{file_id}.h5"
        mask_name = f"mask_{file_id}.h5"

        with h5py.File(os.path.join(self.img_dir, img_name), "r") as f:
            raw_image = f["img"][:]

        if self.mask_dir is not None:
            with h5py.File(os.path.join(self.mask_dir, mask_name), "r") as f:
                mask = f["mask"][:]
        else:
            mask = np.zeros((128, 128), dtype=np.int64)

        eps = 1e-6

        blue = raw_image[:, :, 1]
        green = raw_image[:, :, 2]
        red = raw_image[:, :, 3]
        b5 = raw_image[:, :, 4]
        b6 = raw_image[:, :, 5]
        b7 = raw_image[:, :, 6]
        nir = raw_image[:, :, 7]
        swir1 = raw_image[:, :, 10]
        swir2 = raw_image[:, :, 11]
        slope = raw_image[:, :, 12]
        dem = raw_image[:, :, 13]

        _, _, curvature = compute_topographical_features(dem, slope)

        ndvi = (nir - red) / (nir + red + eps)
        bsi = ((swir1 + red) - (nir + blue)) / ((swir1 + red) + (nir + blue) + eps)
        ndwi = (green - nir) / (green + nir + eps)
        ndmi = (nir - swir1) / (nir + swir1 + eps)
        nbr = (nir - swir2) / (nir + swir2 + eps)

        # axis=-1 means the new axis is added at the END → shape: (128, 128, 17)
        image_17ch = np.stack(
            [dem, slope, curvature, blue, green, red, nir, b5, b6, b7, swir1, swir2, ndvi, bsi, ndwi, ndmi, nbr], axis=-1
        ).astype(np.float32)  # Final shape: (128, 128, 17)

        image_17ch = np.nan_to_num(image_17ch, nan=0.0, posinf=0.0, neginf=0.0)

        if self.transform:

            augmented = self.transform(image=image_17ch, mask=mask)
            image = augmented['image'].float()
            mask = augmented['mask'].long()
        else:
            image_17ch = image_17ch.transpose((2, 0, 1))  # (C, H, W)

            image = torch.from_numpy(image_17ch).float()
            mask = torch.from_numpy(mask).long()

        return image, mask

In [ ]:
# # ----- Part-1: U-Net Architecture -----


# class ConvBlock(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.block = nn.Sequential(
#             nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),  ## out_channels means filters in keras, and keras figures out the in_channel itself
#             nn.BatchNorm2d(out_channels),
#             nn.ReLU(inplace=True),
#             nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
#             nn.BatchNorm2d(out_channels),
#             nn.ReLU(inplace=True),
#         )

#     def forward(self, x):
#         return self.block(x)


# # Encoder Block: ConvBlock + MaxPool


# class EncoderBlock(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.conv = ConvBlock(in_channels, out_channels)
#         self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

#     def forward(self, x):
#         features = self.conv(x)
#         pooled = self.pool(features)
#         return features, pooled
#         # return both:
#         # features -> will be passed accross via skip connection to decoder
#         # pooled -> goes down to the next encoder block


# # Decoder Block: Upsampe + Concatenate skip + convBlock
# class DecoderBlock(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.upsample = nn.ConvTranspose2d(
#             in_channels, out_channels, kernel_size=2, stride=2
#         )
#         # convTransposed2d doubles the spatial size: (64, 64) → (128, 128)

#         self.conv = ConvBlock(out_channels * 2, out_channels)

#     def forward(self, x, skip):
#         x = self.upsample(x)
#         x = torch.cat([x, skip], dim=1)  # concatenate along channel dimension
#         x = self.conv(x)
#         return x


# # --- Full U-Net Model ---
# class UNet(nn.Module):
#     def __init__(self, in_channels=8, num_classes=2):
#         """
#         in_chennels : number of input channels - 8 for our dataset
#         num_classes : 2 for binary segmentation (landslide / no-landslide)
#         """

#         super().__init__()

#         # Encoder ( Contracting Path)

#         self.enc1 = EncoderBlock(in_channels, 64)  # 8 → 64
#         self.enc2 = EncoderBlock(64, 128)  # 64 → 128
#         self.enc3 = EncoderBlock(128, 256)  # 128 → 256
#         self.enc4 = EncoderBlock(256, 512)  # 256 → 512

#         # Bottleneck (deepest point - no pooling)

#         self.bottleneck = ConvBlock(512, 1024)  # 512 → 1024

#         # Decoder (Expanding Path)

#         self.dec4 = DecoderBlock(1024, 512)  # 1024 → 512
#         self.dec3 = DecoderBlock(512, 256)  # 512 → 256
#         self.dec2 = DecoderBlock(256, 128)  # 256 → 128
#         self.dec1 = DecoderBlock(128, 64)  # 128 → 64

#         # --- Final Output Layer ---
#         self.output_conv = nn.Conv2d(64, num_classes, kernel_size=1)
#         # kernel size=1 -> 1x1 convolution, just maps 64 channels -> num_classes

#     def forward(self, x):
#         # ---Encoder ---
#         skip1, x = self.enc1(x)  # skip for dec1, x: goes to enc2
#         skip2, x = self.enc2(x)
#         skip3, x = self.enc3(x)
#         skip4, x = self.enc4(x)  # skip for dec4, x: goes to bottleneck

#         # --- Bottleneck ---
#         x = self.bottleneck(x)

#         # --- Decoder ---
#         x = self.dec4(x, skip4)  # input from bottleneck, skip from enc4
#         x = self.dec3(x, skip3)
#         x = self.dec2(x, skip2)
#         x = self.dec1(x, skip1)  # input from dec2, skip from enc1

#         # --- Final Output ---
#         return self.output_conv(x)  # shape: (Batch, 2, 128, 128)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --------------ResUNet Model----------------------

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        # First convolution
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=False)
        
        # Second convolution
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Shortcut connection
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        out = out + identity  # Out-of-place addition
        out = self.relu(out)
        
        return out


# ---- Encoder Block ---- #

class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = ResidualBlock(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        features = self.conv(x)
        pooled = self.pool(features)
        return features, pooled


# ---- Decoder Block ---- #

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.conv = ResidualBlock(out_channels * 2, out_channels)

    def forward(self, x , skip):
            upsampled = self.upsample(x)
            cat = torch.cat([upsampled, skip], dim=1)
            x = self.conv(cat)
            return x


class ResUNet(nn.Module):
    def __init__(self, in_channels=15, num_classes=2):
        super().__init__()

        # -- Encoding Phase -- 
        self.enc1 = EncoderBlock(in_channels, 64)
        self.enc2 = EncoderBlock(64, 128)
        self.enc3 = EncoderBlock(128, 256)
        self.enc4 = EncoderBlock(256, 512)

        # -- Bottleneck (deepest point - no pooling here) --
        self.bottleneck = ResidualBlock(512, 1024)

        # -- Decoding Phase --
        self.dec4 = DecoderBlock(1024, 512)
        self.dec3 = DecoderBlock(512, 256)
        self.dec2 = DecoderBlock(256, 128)
        self.dec1 = DecoderBlock(128, 64)

        # -- Final Output Layer -- 
        self.output_conv = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        # --- Encoder ---
        skip1, x = self.enc1(x)
        skip2, x = self.enc2(x)
        skip3, x = self.enc3(x)
        skip4, x = self.enc4(x)

        # --- Bottleneck ---
        x = self.bottleneck(x)

        # --- Decoder ---
        x = self.dec4(x, skip4)
        x = self.dec3(x, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)

        # Final output
        return self.output_conv(x)

In [ ]:
# # ---------FUlly Connected Graph Attention Network (GAT) Layer ---------

# import torch.nn.functional as F

# class GATLayer(nn.Module):
#     def __init__(self, in_dim, out_dim, num_heads=4, dropout=0.1, concat=True):
#         super().__init__()
#         self.out_dim = out_dim
#         self.num_heads = num_heads
#         self.concat = concat
        
#         self.W = nn.Linear(in_dim, out_dim * num_heads, bias=False)     
        
#         self.a_src = nn.Parameter(torch.empty(size=(num_heads, out_dim)))
#         self.a_dst = nn.Parameter(torch.empty(size=(num_heads, out_dim)))
#         nn.init.xavier_uniform_(self.a_src)
#         nn.init.xavier_uniform_(self.a_dst)
        
#         self.leakyrelu = nn.LeakyReLU(0.2)
#         self.dropout = nn.Dropout(dropout)
    
#     def forward(self, x, adj=None):
#         B, N, _ = x.shape  # Batch size, Number of nodes, Feature dimension
#         H, D = self.num_heads, self.out_dim
        
#         # step 1: project every node, split into heads -> ( B, N, H, D)
#         Wh = self.W(x).view(B, N, H, D)
        
#         # step 2: compute attention logits e_ij for EvERY pair (i, j), per head, at once.
#         # src_scores = (wh * self.a_scr).sum(dim=-1) # (B, N, H)
#         # dst_scores = (Wh * self.a_dst).sum(dim=-1)  # (B, N, H)
#         src_scores = (Wh * self.a_src).sum(dim=-1)
#         dst_scores = (Wh * self.a_dst).sum(dim=-1)
        
#         e = src_scores.unsqueeze(2) + dst_scores.unsqueeze(1)  # (B, N, N, H)
#         e = self.leakyrelu(e)
        
#         # step 3: normalize into proper attention weight (softmax over neighbors j,
#         # for each destination node i)- every node's incomming weights sum to 1 
#         alpha = torch.softmax(e, dim=2) # (B, N, N, H)
#         alpha = self.dropout(alpha)
        
#         # step 4: weighted aggregation - out_i = sum_j(alpha_ij * Wh_j)
#         out = torch.einsum('bijh,bjhd->bihd', alpha, Wh)  # (B, N, H, D)
        
#         if self.concat:
#             out = out.reshape(B, N, H * D)  # (B, N, H*D)
#         else:
#             out = out.mean(dim=2)  # (B, N, D)
#         return out
    
    
# class SpatialGATBlock(nn.Module):
#     def __init__(self, channels, num_heads=4, num_layers=2, dropout=0.1):
#         super().__init__()
#         head_dim = channels // num_heads
#         assert head_dim * num_heads == channels, "channels must be divisible by num_heads"
        
#         self.layers = nn.ModuleList([
#             GATLayer(channels, head_dim, num_heads=num_heads, dropout=dropout, concat=True)
#             for _ in range(num_layers)
#         ])  
#         self.norm = nn.LayerNorm(channels)
        
#     def forward(self, x, adj=None):
#         B, C, H, W = x.shape
#         N = H * W
        
#         # (B, C, H, W) -> (B, N, C): one row per spatial location = one graph node
#         nodes = x.flatten(2).transpose(1, 2)  # (B, N, C)
        
#         h = nodes
#         for layer in self.layers:
#             h = layer(h, adj=adj)
#             h = F.elu(h)        
            
#         h = self.norm(nodes + h)
#         out = h.transpose(1, 2).reshape(B, C, H, W) 
#         return out
    


# class ResidualBlock(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
        
#         # First convolution
#         self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
#         self.bn1 = nn.BatchNorm2d(out_channels)
#         self.relu = nn.ReLU(inplace=True)
        
#         # Second convolution
#         self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
#         self.bn2 = nn.BatchNorm2d(out_channels)
        
#         # Shortcut connection
#         # if the input and the output dont match, we need a 1x1 convolution,
#         # to project the correct input before adding.
#         self.shortcut = nn.Sequential()
#         if in_channels != out_channels:
#             self.shortcut = nn.Sequential(
#                 nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
#                 nn.BatchNorm2d(out_channels)
#             )
            

#     def forward(self, x):
#         identity = self.shortcut(x)
        
#         out = self.conv1(x); out = self.bn1(out); out = self.relu(out)
        
#         out = self.conv2(out); out = self.bn2(out)
        
#         out += identity
#         out = self.relu(out)
        
#         return out


# # ---- Encoder Block ---- #

# class EncoderBlock(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.conv = ResidualBlock(in_channels, out_channels)
#         self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

#     def forward(self, x):
#         features = self.conv(x)
#         pooled = self.pool(features)
#         return features, pooled

# # ---- Decoder Block ---- #

# class DecoderBlock(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.upsample = nn.ConvTranspose2d(
#             in_channels, out_channels, kernel_size=2, stride=2
#         )

#         self.conv = ResidualBlock(out_channels * 2, out_channels)

#     def forward(self, x , skip):
#         upsampled = self.upsample(x)
#         cat = torch.cat([upsampled, skip], dim=1)
#         x = self.conv(cat)
#         return x
    
# class ResUNetGAT(nn.Module):
#     def __init__(self, in_channels=17, num_classes=2, gat_heads=4, gat_layers=2):
#         super().__init__()

#         # --Encoding Phase -- 

#         self.enc1 = EncoderBlock(in_channels, 64)
#         self.enc2 = EncoderBlock(64, 128)
#         self.enc3 = EncoderBlock(128, 256)
#         self.enc4 = EncoderBlock(256, 512)

#         # --Bottleneck (deepest point - no pooling here)

#         self.bottleneck = ResidualBlock(512, 1024)
        
#         # --GAT Block-- global, non-local reasoning over the 8x8 = 64 bottleneck
#         self.gat_block = SpatialGATBlock(channels=1024, num_heads=gat_heads, num_layers=gat_layers)

#         # --Decoding Phase --

#         self.dec4 = DecoderBlock(1024, 512)
#         self.dec3 = DecoderBlock(512, 256)
#         self.dec2 = DecoderBlock(256, 128)
#         self.dec1 = DecoderBlock(128, 64)

#         # -- Final Outout Layer -- 
#         self.output_conv = nn.Conv2d(64, num_classes, kernel_size=1)

#     def forward(self, x):
#             # ---Encoder---
#         skip1, x = self.enc1(x)
#         skip2, x = self.enc2(x)
#         skip3, x = self.enc3(x)
#         skip4, x = self.enc4(x)

#         # ---Bottleneck + GAT---
#         x = self.bottleneck(x)
#         x = self.gat_block(x)      

#         #---Decoder---
#         x = self.dec4(x, skip4)
#         x = self.dec3(x, skip3)
#         x = self.dec2(x, skip2)
#         x = self.dec1(x, skip1)

#         # Final output

#         return self.output_conv(x)
        

In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# # =============================================================================
# # PART 3: GRAPH ATTENTION NETWORK (GAT) BOTTLENECK
# # =============================================================================
# # BACKGROUND CONTEXT — what a GAT actually is:
# #
# # A CNN convolution aggregates information from a small, FIXED neighborhood
# # (e.g. a 3x3 window) using the SAME learned weights everywhere in the image.
# # It has no way to say "this pixel is irrelevant to me" or "that pixel, way
# # over there, actually matters a lot" — every neighbor gets the same fixed
# # kernel treatment regardless of what's actually in the image.
# #
# # A Graph Neural Network (GNN) generalizes "convolution" to work over an
# # arbitrary graph: a set of nodes, each with a feature vector, connected by
# # edges. A GNN layer updates every node's features by aggregating (some
# # combination of) its neighbors' features — this is called "message passing".
# #
# #   - Graph Convolutional Network (GCN): aggregates neighbors with weights
# #     fixed by the graph structure alone (basically a normalized average).
# #     Every neighbor is treated as equally important.
# #
# #   - Graph ATTENTION Network (GAT, Velickovic et al. 2018): learns, for
# #     every edge (i, j), an attention weight alpha_ij that depends on BOTH
# #     node i's and node j's actual features. So instead of "average my
# #     neighbors", it's "let me learn how much to listen to each neighbor,
# #     based on what's actually in their features right now."
# #
# #     e_ij       = LeakyReLU( a_src . (W h_i) + a_dst . (W h_j) )   <- raw score
# #     alpha_ij   = softmax_j( e_ij )                                <- normalize per node i
# #     h_i_new    = sum_j ( alpha_ij * (W h_j) )                     <- weighted aggregation
# #
# #     This is repeated with several independent "heads" (multi-head
# #     attention, exactly like a Transformer) and the results are concatenated
# #     — different heads can learn to attend to different kinds of
# #     relationships.
# #
# # WHY THIS FITS AT THE RESUNET BOTTLENECK:
# # Your bottleneck feature map is (Batch, 1024, 8, 8) for 128x128 input patches
# # (128 -> 64 -> 32 -> 16 -> 8 after 4 pooling stages). That's only 8*8 = 64
# # spatial locations. We treat each of those 64 locations as one graph "node"
# # with a 1024-dim feature vector, and let every node attend to every other
# # node (a fully-connected graph — cheap and simple at N=64: only 64*64=4096
# # pairs per head, trivial compute).
# #
# # This gives the model genuine GLOBAL reasoning that convolutions structurally
# # cannot provide, even at the bottleneck: a landslide signature on one side of
# # a ridge can directly inform what the model predicts on the other side, or
# # a scar downslope can be linked to the terrain that fed it, regardless of
# # raw pixel distance. This bottleneck placement (replacing/augmenting the
# # deepest conv block with GAT layers) is the standard way this has been done
# # in published CNN+GAT segmentation hybrids — the same pattern published
# # GAT-based landslide segmentation work (object/graph-based landslide models)
# # and general "GNN at the U-Net bottleneck" architectures both use.
# # =============================================================================


# def build_grid_adjacency(H, W, connectivity=8, self_loops=True):
#     """
#     Builds a spatially-local adjacency matrix: node i is only connected to its
#     immediate spatial neighbours (8-connectivity by default: up/down/left/right
#     + diagonals; use connectivity=4 for just up/down/left/right). By default,
#     each node is also connected to itself, matching the standard GAT setup.

#     This matters when graph_type="local": instead of a node having to learn,
#     across a nearly-empty-of-signal 64-node graph, which of 63 other nodes to
#     ignore, it starts with a strong structural prior — "your neighbours are
#     the locations physically next to you" — and only has to learn HOW MUCH to
#     weight each of those ~8, not WHICH ones might matter out of 64. Given how
#     sparse landslide-positive pixels are in this dataset, that's a much easier
#     thing to learn well from limited data than dense attention is.
#     """
#     N = H * W
#     adj = torch.zeros(N, N)
#     for i in range(H):
#         for j in range(W):
#             idx = i * W + j
#             for di in (-1, 0, 1):
#                 for dj in (-1, 0, 1):
#                     if di == 0 and dj == 0:
#                         continue
#                     if connectivity == 4 and abs(di) + abs(dj) != 1:
#                         continue
#                     ni, nj = i + di, j + dj
#                     if 0 <= ni < H and 0 <= nj < W:
#                         adj[idx, ni * W + nj] = 1
#     if self_loops:
#         adj.fill_diagonal_(1)
#     return adj


# class GATLayer(nn.Module):
#     """
#     A single Graph Attention layer, implemented from scratch (no torch_geometric
#     dependency needed — at N=64 nodes, a hand-rolled dense implementation is
#     simpler to install and just as fast as a sparse graph library).

#     Operates on a "fully-connected" graph by default: every node can attend to
#     every other node. Pass an `adj` mask if you want to restrict this later
#     (e.g. to a local spatial neighborhood, or a slope/DEM-informed terrain graph).
#     """
#     def __init__(self, in_dim, out_dim, num_heads=4, dropout=0.1, concat=True):
#         super().__init__()
#         self.num_heads = num_heads
#         self.out_dim = out_dim
#         self.concat = concat

#         # Shared linear projection W, applied to every node (same weights for all nodes,
#         # exactly like a conv kernel is shared across all spatial positions).
#         # Projects into (num_heads * out_dim) so we can split into heads afterwards.
#         self.W = nn.Linear(in_dim, out_dim * num_heads, bias=False)

#         # Attention scoring parameters. The original GAT paper scores a pair (i, j) with
#         # a single vector 'a' applied to the concatenation [Wh_i || Wh_j]. We split that
#         # into two separate learnable vectors (a_src, a_dst) so the score can be computed
#         # as a_src.Wh_i + a_dst.Wh_j — mathematically equivalent, but lets us compute all
#         # N*N pairs at once via broadcasting instead of an explicit N*N concatenation.
#         self.a_src = nn.Parameter(torch.empty(num_heads, out_dim))
#         self.a_dst = nn.Parameter(torch.empty(num_heads, out_dim))
#         nn.init.xavier_uniform_(self.a_src)
#         nn.init.xavier_uniform_(self.a_dst)

#         self.leaky_relu = nn.LeakyReLU(0.2)   # matches the GAT paper's choice
#         self.dropout = nn.Dropout(dropout)

#     def forward(self, x, adj=None):
#         # ---------------------------------------------------------------
#         # INPUT SHAPES:
#         # x:   (Batch, N, in_dim)  -> N graph nodes, each an in_dim feature vector
#         # adj: (N, N) or None      -> 1 where an edge exists; None = fully connected
#         # ---------------------------------------------------------------
#         B, N, _ = x.shape
#         H, D = self.num_heads, self.out_dim

#         # STEP 1: project every node, split into heads -> (B, N, H, D)
#         Wh = self.W(x).view(B, N, H, D)

#         # STEP 2: compute attention logits e_ij for EVERY pair (i, j), per head, at once.
#         # src_scores[i] = "how much node i has to offer as a neighbor"
#         # dst_scores[j] = "how much node j is looking for that kind of info"
#         src_scores = (Wh * self.a_src).sum(dim=-1)          # (B, N, H)
#         dst_scores = (Wh * self.a_dst).sum(dim=-1)          # (B, N, H)

#         # Broadcasting trick: e[:, i, j, :] = src_scores[:, i, :] + dst_scores[:, j, :]
#         e = src_scores.unsqueeze(2) + dst_scores.unsqueeze(1)   # (B, N, N, H)
#         e = self.leaky_relu(e)

#         # STEP 3 (optional): mask out non-edges before softmax, so disconnected
#         # nodes get exactly zero attention weight instead of a small nonzero one.
#         if adj is not None:
#             mask = (adj == 0).unsqueeze(0).unsqueeze(-1)     # (1, N, N, 1)
#             e = e.masked_fill(mask, float('-1e9'))

#         # STEP 4: normalize into proper attention weights (softmax over neighbors j,
#         # for each destination node i) — every node's incoming weights sum to 1.
#         alpha = torch.softmax(e, dim=2)      # (B, N, N, H)
#         alpha = self.dropout(alpha)

#         # STEP 5: weighted aggregation — out_i = sum_j( alpha_ij * Wh_j )
#         out = torch.einsum('bijh,bjhd->bihd', alpha, Wh)     # (B, N, H, D)

#         if self.concat:
#             out = out.reshape(B, N, H * D)      # concat heads, like multi-head attention
#         else:
#             out = out.mean(dim=2)               # average heads (used for a final layer)

#         return out


# class SpatialGATBlock(nn.Module):
#     """
#     Wraps GATLayer(s) so they can drop straight into a CNN pipeline: takes a
#     (Batch, Channels, Height, Width) feature map, treats every spatial location
#     as one graph node, runs `num_layers` stacked GAT layers over the fully-
#     connected graph of all locations, and reshapes back to (B, C, H, W).

#     A residual connection + LayerNorm wraps the whole thing (same pattern as
#     a Transformer block) — this matters early in training: the GAT starts out
#     producing near-random output, and the residual connection guarantees the
#     original conv features still flow through untouched while the GAT slowly
#     learns something useful to *add* on top, instead of the model having to
#     relearn everything from scratch through the GAT.
#     """
#     def __init__(self, channels, num_heads=4, num_layers=2, dropout=0.1,
#                  graph_type="dense", spatial_size=(8, 8)):
#         super().__init__()
#         head_dim = channels // num_heads
#         assert head_dim * num_heads == channels, \
#             "channels must be divisible by num_heads (1024 / 4 = 256, for example)"

#         self.layers = nn.ModuleList([
#             GATLayer(channels, head_dim, num_heads=num_heads, dropout=dropout, concat=True)
#             for _ in range(num_layers)
#         ])
#         self.norm = nn.LayerNorm(channels)
#         self.gat_scale = nn.Parameter(torch.tensor(0.1))

#         # graph_type="dense"  -> every node attends to every other node (original default)
#         # graph_type="local"  -> each node only attends to its 8 spatial neighbours,
#         #                        precomputed ONCE here and reused every forward pass
#         #                        (register_buffer -> moves with .to(device), not a param)
#         self.graph_type = graph_type
#         if graph_type == "local":
#             H, W = spatial_size
#             self.register_buffer("adj", build_grid_adjacency(H, W, connectivity=8, self_loops=True))
#         else:
#             self.adj = None

#     def forward(self, x):
#         B, C, H, W = x.shape
#         N = H * W

#         if self.adj is not None and self.adj.shape[0] != N:
#             raise ValueError(f"GAT adjacency has {self.adj.shape[0]} nodes, but bottleneck has {N} nodes")

#         # (B, C, H, W) -> (B, N, C): one row per spatial location = one graph node
#         nodes = x.flatten(2).transpose(1, 2)     # (B, N, C)

#         h = nodes
#         for layer in self.layers:
#             h = layer(h, adj=self.adj)
#             h = F.elu(h)

#         # Keep the original ResUNet bottleneck path dominant at the start; the
#         # GAT branch learns as a small residual correction instead of replacing it.
#         h = nodes + self.gat_scale * self.norm(h)
#         out = h.transpose(1, 2).reshape(B, C, H, W)
#         return out


# # =============================================================================
# # PART 4: RESUNET WITH GAT INTEGRATED AT THE BOTTLENECK
# # =============================================================================
# # Only two lines change relative to your original ResUNet:
# #   1. __init__:  add `self.gat_block = SpatialGATBlock(...)` after `self.bottleneck`
# #   2. forward:   call `x = self.gat_block(x)` right after `x = self.bottleneck(x)`,
# #                 before it goes into dec4. Everything else — encoder, decoder,
# #                 skip connections, output layer — is untouched.
# # =============================================================================

# class ResidualBlock(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
#         self.bn1 = nn.BatchNorm2d(out_channels)
#         self.relu = nn.ReLU(inplace=True)
#         self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
#         self.bn2 = nn.BatchNorm2d(out_channels)
#         self.shortcut = nn.Sequential()
#         if in_channels != out_channels:
#             self.shortcut = nn.Sequential(
#                 nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
#                 nn.BatchNorm2d(out_channels)
#             )

#     def forward(self, x):
#         identity = self.shortcut(x)
#         out = self.conv1(x); out = self.bn1(out); out = self.relu(out)
#         out = self.conv2(out); out = self.bn2(out)
#         out += identity
#         return self.relu(out)


# class EncoderBlock(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.conv = ResidualBlock(in_channels, out_channels)
#         self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

#     def forward(self, x):
#         features = self.conv(x)
#         pooled = self.pool(features)
#         return features, pooled


# class DecoderBlock(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.upsample = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
#         self.conv = ResidualBlock(out_channels * 2, out_channels)

#     def forward(self, x, skip):
#         upsampled = self.upsample(x)
#         cat = torch.cat([upsampled, skip], dim=1)
#         return self.conv(cat)


# class ResUNetGAT(nn.Module):
#     def __init__(self, in_channels=17, num_classes=2, gat_heads=4, gat_layers=2,
#                  graph_type="local", spatial_size=(8, 8)):
#         super().__init__()

#         # --Encoding Phase--
#         self.enc1 = EncoderBlock(in_channels, 64)
#         self.enc2 = EncoderBlock(64, 128)
#         self.enc3 = EncoderBlock(128, 256)
#         self.enc4 = EncoderBlock(256, 512)

#         # --Bottleneck-- (deepest point, no pooling here)
#         self.bottleneck = ResidualBlock(512, 1024)

#         # --GAT block-- reasoning over the 8x8 = 64 bottleneck locations, right
#         # after the conv bottleneck and before decoding starts.
#         # graph_type="dense" -> every node attends to every other node (64 edges/node)
#         # graph_type="local" -> each node only attends to its 8 spatial neighbours
#         self.gat_block = SpatialGATBlock(channels=1024, num_heads=gat_heads, num_layers=gat_layers,
#                                           graph_type=graph_type, spatial_size=spatial_size)

#         # --Decoding Phase--
#         self.dec4 = DecoderBlock(1024, 512)
#         self.dec3 = DecoderBlock(512, 256)
#         self.dec2 = DecoderBlock(256, 128)
#         self.dec1 = DecoderBlock(128, 64)

#         # --Final Output Layer--
#         self.output_conv = nn.Conv2d(64, num_classes, kernel_size=1)

#     def forward(self, x):
#         # ---Encoder---
#         skip1, x = self.enc1(x)
#         skip2, x = self.enc2(x)
#         skip3, x = self.enc3(x)
#         skip4, x = self.enc4(x)

#         # ---Bottleneck + GAT---
#         x = self.bottleneck(x)
#         x = self.gat_block(x)          # <-- THIS is where the GAT sits

#         # ---Decoder---
#         x = self.dec4(x, skip4)
#         x = self.dec3(x, skip3)
#         x = self.dec2(x, skip2)
#         x = self.dec1(x, skip1)

#         return self.output_conv(x)


# # =============================================================================
# # SANITY CHECK — confirms shapes flow correctly and gradients reach every
# # parameter, including the GAT block, before you drop this into real training.
# # # =============================================================================
# # if __name__ == "__main__":
# #     model = ResUNetGAT(in_channels=17, num_classes=2, gat_heads=4, gat_layers=2,
# #                        graph_type="local", spatial_size=(8, 8))

# #     x = torch.randn(2, 17, 128, 128)          # (Batch=2, Channels=17, H=128, W=128)
# #     out = model(x)
# #     print("input :", tuple(x.shape))
# #     print("output:", tuple(out.shape))
# #     assert out.shape == (2, 2, 128, 128)

# #     out.sum().backward()                       # confirm gradients flow end-to-end
# #     print("backward pass OK — no NaNs:", not torch.isnan(out).any().item())

# #     total_params = sum(p.numel() for p in model.parameters())
# #     gat_params = sum(p.numel() for p in model.gat_block.parameters())
# #     print(f"total params: {total_params:,}")
# #     print(f"GAT block params: {gat_params:,} ({100*gat_params/total_params:.1f}% of the model)")

In [ ]:
import os
import h5py
import numpy as np
import torch
import torch.nn as nn


class DiceLoss(nn.Module):
    """Unchanged from your version — ignores background, maximizes overlap on
    the landslide class."""
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, predictions, targets):
        probs = torch.softmax(predictions, dim=1)[:, 1, :, :]
        targets_f = targets.float()

        intersection = (probs * targets_f).sum(dim=(1, 2))
        dice = (2.0 * intersection + self.smooth) / (
            probs.sum(dim=(1, 2)) + targets_f.sum(dim=(1, 2)) + self.smooth
        )
        return 1 - dice.mean()


class BinaryFocalLoss(nn.Module):
    """Unchanged from your version. Kept here as an alternative to try —
    if you use it, set alpha close to (1 - positive_pixel_fraction), see
    compute_class_weights below for how to get that fraction."""
    def __init__(self, alpha, gamma, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)[:, 1]
        targets_f = targets.float()

        pt = torch.where(targets_f == 1, probs, 1 - probs)
        alpha_t = torch.where(targets_f == 1, self.alpha, 1 - self.alpha)
        focal = -alpha_t * (1 - pt) ** self.gamma * torch.log(pt + self.smooth)
        return focal.mean()


class CombinedFocalDiceLoss(nn.Module):
    def __init__(self, focal_weight, dice_weight, alpha, gamma):
        super().__init__()
        self.focal = BinaryFocalLoss(alpha=alpha, gamma=gamma)
        self.dice = DiceLoss()
        self.focal_weight = focal_weight
        self.dice_weight = dice_weight

    def forward(self, predictions, targets):
        return (self.focal_weight * self.focal(predictions, targets) +
                self.dice_weight * self.dice(predictions, targets))


class CombinedLoss(nn.Module):
    """CE + Dice, same as your version, but CE now accepts optional per-class
    weights. Unweighted CE still lets background dominate the gradient even
    when blended with Dice — this is the fix for that."""
    def __init__(self, dice_weight=0.5, ce_weight=0.5, class_weights=None):
        super().__init__()
        self.dice_weight = dice_weight
        self.ce_weight = ce_weight
        self.dice = DiceLoss()
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def forward(self, predictions, targets):
        return (self.ce_weight * self.ce(predictions, targets) +
                self.dice_weight * self.dice(predictions, targets))


def compute_class_weights(mask_dir, file_ids, num_classes=2, dampen="sqrt"):
    """Inverse-frequency weights computed from actual pixel counts in the
    training masks.

    IMPORTANT: on data this imbalanced (landslide pixels are often only
    ~4-5% of all pixels), the RAW inverse-frequency ratio comes out extreme
    (~20-40x). Applying that directly as a per-pixel CE weight, on top of
    Dice loss (which already ignores background and rewards only landslide
    overlap) AND a WeightedRandomSampler that already oversamples positive
    patches, triple-corrects for the same imbalance. The three stack and
    push the model to massively over-predict the positive class — you'll
    see this as recall near 1.0 but precision collapsing toward 0.2-0.3.

    `dampen="sqrt"` (default) compresses the ratio (e.g. 21.6x -> ~4.6x) so
    CE still nudges the model in the right direction without dominating the
    gradient. Set dampen=None to get the old raw-ratio behavior back.
    """
    class_counts = np.zeros(num_classes, dtype=np.float64)

    for file_id in file_ids:
        mask_path = os.path.join(mask_dir, f"mask_{file_id}.h5")
        if not os.path.exists(mask_path):
            continue
        with h5py.File(mask_path, "r") as f:
            mask = f["mask"][:]
        for c in range(num_classes):
            class_counts[c] += (mask == c).sum()

    total = class_counts.sum()
    raw_weights = total / (num_classes * class_counts + 1e-6)

    if dampen == "sqrt":
        weights = np.sqrt(raw_weights)
    elif dampen == "log":
        weights = np.log1p(raw_weights)
    else:
        weights = raw_weights

    return torch.as_tensor(weights, dtype=torch.float32)

In [ ]:
import numpy as np
import torch


def compute_metrics(predictions, targets, threshold=0.5):
    probs = torch.softmax(predictions, dim=1)[:, 1]
    pred_bin = probs > threshold

    tp = ((targets == 1) & (pred_bin == 1)).sum().float()
    fp = ((targets == 0) & (pred_bin == 1)).sum().float()
    fn = ((targets == 1) & (pred_bin == 0)).sum().float()
    return tp, fp, fn


@torch.no_grad()
def evaluate(model, dataloader, criterion, device, threshold=0.5):
    model.eval()
    running_loss = 0.0
    total_tp, total_fp, total_fn = 0.0, 0.0, 0.0

    for images, targets in dataloader:
        images, targets = images.to(device), targets.to(device)
        predictions = model(images)

        if criterion:
            loss = criterion(predictions, targets)
            running_loss += loss.item()

        tp, fp, fn = compute_metrics(predictions, targets, threshold=threshold)
        total_tp += tp.item()
        total_fp += fp.item()
        total_fn += fn.item()

    avg_loss = running_loss / len(dataloader) if criterion else 0.0
    iou = total_tp / (total_tp + total_fp + total_fn + 1e-6)
    f1 = 2 * total_tp / (2 * total_tp + total_fp + total_fn + 1e-6)
    precision = total_tp / (total_tp + total_fp + 1e-6)
    recall = total_tp / (total_tp + total_fn + 1e-6)

    return {"loss": avg_loss, "iou": iou, "f1": f1, "precision": precision, "recall": recall}


def find_best_threshold(model, dataloader, device, thresholds=None):
    """Sweep decision thresholds on validation data, return the one that
    maximizes F1. With this much imbalance the optimal cut is usually well
    under 0.5 — free performance, no retraining."""
    if thresholds is None:
        thresholds = np.arange(0.10, 0.91, 0.05)

    best_f1, best_threshold = -1.0, 0.5
    sweep_results = []
    for t in thresholds:
        m = evaluate(model, dataloader, criterion=None, device=device, threshold=float(t))
        sweep_results.append((float(t), m["f1"], m["iou"]))
        if m["f1"] > best_f1:
            best_f1 = m["f1"]
            best_threshold = float(t)

    return best_threshold, best_f1, sweep_results

In [ ]:
import os

# Must be set before HDF5 initializes (i.e. before h5py is imported anywhere
# in the process). h5py + PyTorch's multi-worker DataLoader is a known
# combination that can silently deadlock on Linux: HDF5's own file-locking
# interacts badly with forked worker processes, especially when the main
# process has already opened many .h5 files (which compute_normalization and
# compute_sample_weights do, right before your workers fork). This disables
# that locking. If you're running this in a notebook, put these two lines in
# your VERY FIRST cell, before any other import.
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")

import random
import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler

# =============================================================================
# CHANGES vs your original train_from_scratch:
#
#  1. AdamW instead of Adam           — decoupled weight decay, standard upgrade
#  2. WeightedRandomSampler           — oversamples patches that contain
#                                        landslide pixels so batches aren't
#                                        dominated by pure-background patches
#  3. Per-class weighted CE + Dice    — CE now actually accounts for the real
#                                        pixel imbalance (computed from data)
#  4. Gradient clipping (max_norm=1)  — the most likely fix for the epoch-to-
#                                        epoch F1 whiplash you were seeing
#  5. Mixed precision (torch.cuda.amp) — faster iteration, same accuracy
#  6. Scheduler watches val F1        — matches what you checkpoint/early-stop
#     (mode='max')                      on, instead of watching val loss
#  7. Threshold sweep after training  — picks the F1-optimal decision
#                                        threshold on validation, applied to test
#  8. Optional TTA on the final test  — flip/rotate ensemble, free ~1-3 F1 pts
# =============================================================================


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def train_from_scratch(
    train_img_dir,
    train_mask_dir,
    val_img_dir,
    val_mask_dir,
    test_img_dir,
    test_mask_dir,
    epochs,
    batch_size,
    save_path,
    patience=15,
    resume=False,
    seed=42,
    num_workers=2,
    use_amp=True,
    use_tta_on_test=True,
):
    set_seed(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = use_amp and device.type == "cuda"
    print(f"Using {device} for training! Seed: {seed} | Workers: {num_workers} | AMP: {use_amp}")

    train_ids = [int(f.split("_")[1].split(".")[0]) for f in os.listdir(train_img_dir) if f.endswith(".h5")]
    val_ids = [int(f.split("_")[1].split(".")[0]) for f in os.listdir(val_img_dir) if f.endswith(".h5")]
    test_ids = [int(f.split("_")[1].split(".")[0]) for f in os.listdir(test_img_dir) if f.endswith(".h5")]

    print(f"Train samples: {len(train_ids)} | Val samples: {len(val_ids)} | Test samples: {len(test_ids)}")

    print("\n---- Computing Normalization Statistics From Training Split ----")
    MEANS, STDS = compute_normalization(train_img_dir, train_ids)
    print(f"Means: {MEANS}")
    print(f"Stds:  {STDS}")

    print("\n---- Computing Per-Sample Weights For Oversampling ----")
    sample_weights, n_pos, n_neg = compute_sample_weights(train_mask_dir, train_ids)
    print(f"Patches with landslide pixels: {n_pos} | without: {n_neg}")

    print("\n---- Computing Class Weights For Weighted Cross-Entropy ----")
    class_weights = compute_class_weights(train_mask_dir, train_ids, dampen="sqrt").to(device)
    print(f"Class weights [background, landslide] (sqrt-dampened): {class_weights.tolist()}")

    train_dataset = LandslideDataset(train_img_dir, train_mask_dir, transform=train_transform(MEANS, STDS), file_ids=train_ids)
    val_dataset = LandslideDataset(val_img_dir, val_mask_dir, transform=val_transform(MEANS, STDS), file_ids=val_ids)
    test_dataset = LandslideDataset(test_img_dir, test_mask_dir, transform=val_transform(MEANS, STDS), file_ids=test_ids)

    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

    loader_kwargs = dict(num_workers=num_workers, pin_memory=(device.type == "cuda"))
    if num_workers > 0:
        loader_kwargs["persistent_workers"] = True

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, **loader_kwargs)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, **loader_kwargs)

    model = ResUNet(in_channels=17, num_classes=2).to(device)
    criterion = CombinedLoss(dice_weight=0.6, ce_weight=0.4, class_weights=None)

    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=75, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    start_epoch = 1
    best_val_f1 = 0.0
    epochs_without_improvement = 0

    if resume and os.path.exists(save_path):
        print(f"\n[INFO] Resuming training from checkpoint: {save_path}")
        checkpoint = torch.load(save_path, map_location=device)      
        model.load_state_dict(checkpoint["model_state_dict"])
        if "optimizer_state_dict" in checkpoint:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        if "scheduler_state_dict" in checkpoint:
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        if "epoch" in checkpoint:
            start_epoch = checkpoint["epoch"] + 1
        if "best_val_f1" in checkpoint:
            best_val_f1 = checkpoint["best_val_f1"]
        print(f"[INFO] Resuming from Epoch {start_epoch} with Best Val F1: {best_val_f1:.4f}\n")

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        running_train_loss = 0.0

        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()

            with torch.amp.autocast("cuda", enabled=use_amp):
                predictions = model(images)
                loss = criterion(predictions, targets)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            running_train_loss += loss.item()

        train_loss = running_train_loss / len(train_loader)

        val_metrics = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        print(
            f"Epoch [{epoch:02d}/{epochs}] "
            f"| Train Loss: {train_loss:.4f} | Val Loss: {val_metrics['loss']:.4f} "
            f"| Val IoU: {val_metrics['iou']:.4f} | Val F1: {val_metrics['f1']:.4f} "
            f"| Precision: {val_metrics['precision']:.4f} | Recall: {val_metrics['recall']:.4f} "
            f"| LR: {optimizer.param_groups[0]['lr']:.6f}"
        )

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            epochs_without_improvement = 0
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_f1": best_val_f1,
            }, save_path)
            print(f" => Saved new best model checkpoint! F1: {best_val_f1:.4f}")
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"\n[INFO] Early stopping triggered at epoch {epoch}")
            break

    if os.path.exists(save_path):
        checkpoint = torch.load(save_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])

    print("\n---- Sweeping Decision Threshold On Validation Set ----")
    best_threshold, best_val_f1_at_t, sweep_results = find_best_threshold(model, val_loader, device)
    print(f"Best threshold: {best_threshold:.2f} (Val F1 at this threshold: {best_val_f1_at_t:.4f})")
    for t, f1, iou in sweep_results:
        print(f"    threshold={t:.2f} -> F1={f1:.4f} IoU={iou:.4f}")
    
    print("\n---- Final Test Evaluation (tuned threshold) ----")
    test_metrics = evaluate(model, test_loader, criterion=None, device=device, threshold=best_threshold)

    print("\n" + "=" * 55)
    print(
        f"Final Test Metrics -> IoU: {test_metrics['iou']:.4f} | F1: {test_metrics['f1']:.4f} "
        f"| Precision: {test_metrics['precision']:.4f} | Recall: {test_metrics['recall']:.4f}"
    )
    print("=" * 55)

    return model, best_threshold

In [ ]:
# ==========================================
# CELL A: TRAIN FROM SCRATCH
# ==========================================
TRAIN_IMG_DIR  = "/content/landslide4sense/TrainData/img"
TRAIN_MASK_DIR = "/content/landslide4sense/TrainData/mask"
VAL_IMG_DIR  = "/content/landslide4sense/ValidData/img"
VAL_MASK_DIR = "/content/landslide4sense/ValidData/mask"
TEST_IMG_DIR  = "/content/landslide4sense/TestData/img"
TEST_MASK_DIR = "/content/landslide4sense/TestData/mask"
SAVE_PATH = "/content/landslide4sense_model.pth" 

BATCH_SIZE = 16  
EPOCHS     = 75
LEARNING_RATE = 1e-4

print("--- Starting Cloud Landslide Mapping Pipeline ---")


trained_model = train_from_scratch(
    train_img_dir    =  TRAIN_IMG_DIR,
    train_mask_dir   =  TRAIN_MASK_DIR,
    val_img_dir      =  VAL_IMG_DIR,
    val_mask_dir     =  VAL_MASK_DIR,
    test_img_dir     =  TEST_IMG_DIR,
    test_mask_dir    =  TEST_MASK_DIR,
    epochs           =  EPOCHS,
    batch_size =        BATCH_SIZE,
    save_path  =        SAVE_PATH,
    resume     =        False       
)

## Training curves from the text logs

This section compares the training and validation loss curves and the validation IoU curves for the three runs.

In [ ]:
import os
import re
from pathlib import Path
import matplotlib.pyplot as plt


def parse_training_log(log_path):
    pattern = re.compile(
        r"Epoch \[(\d+)/\d+\]\s*\|\s*Train Loss:\s*([0-9.]+)\s*\|\s*Val Loss:\s*([0-9.]+)\s+IoU:\s*([0-9.]+)",
        re.IGNORECASE,
    )

    epochs = []
    train_loss = []
    val_loss = []
    val_iou = []

    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            match = pattern.search(line)
            if match:
                epochs.append(int(match.group(1)))
                train_loss.append(float(match.group(2)))
                val_loss.append(float(match.group(3)))
                val_iou.append(float(match.group(4)))

    return {
        "epochs": epochs,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_iou": val_iou,
    }


base_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
log_files = {
    "UNet-only": "Unetonly.txt",
    "ResUNet-only": "resunetonly.txt",
    "ResUNet-17ch": "resunet_17ch.txt",
    "ResUNet-GAT": "resunet_gat.txt",
    
}

parsed_logs = {}
for label, filename in log_files.items():
    for base in base_candidates:
        path = base /"assets" / filename
        if path.exists():
            parsed_logs[label] = parse_training_log(path)
            break

if not parsed_logs:
    raise FileNotFoundError("Could not find any of the training log files in the current workspace.")

fig, axes = plt.subplots(1, 1, figsize=(7, 5), constrained_layout=True)

for label, data in parsed_logs.items():
    axes.plot(data["epochs"], data["val_iou"], label=label, linewidth=1)


axes.set_title("Validation IoU vs. Epoch")
axes.set_xlabel("Epoch")
axes.set_ylabel("Validation IoU")
axes.grid(True, alpha=0.3)
axes.legend()

plt.tight_layout()
plt.savefig("validation_iou.pdf", format="pdf", dpi=300, bbox_inches="tight", pad_inches=0.02)
plt.savefig("validation_iou.png", dpi=300, bbox_inches="tight", pad_inches=0.02)
plt.show()

In [ ]:
import os
import re
from pathlib import Path
import matplotlib.pyplot as plt


def parse_training_log(log_path):
    pattern = re.compile(
        r"Epoch \[(\d+)/\d+\]\s*\|\s*Train Loss:\s*([0-9.]+)\s*\|\s*Val Loss:\s*([0-9.]+)\s+IoU:\s*([0-9.]+)",
        re.IGNORECASE,
    )

    epochs = []
    train_loss = []
    val_loss = []
    val_iou = []

    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            match = pattern.search(line)
            if match:
                epochs.append(int(match.group(1)))
                train_loss.append(float(match.group(2)))
                val_loss.append(float(match.group(3)))
                val_iou.append(float(match.group(4)))

    return {
        "epochs": epochs,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_iou": val_iou,
    }


base_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
log_files = {
    "UNet-only": "Unetonly.txt",
    "ResUNet-only": "resunetonly.txt",
    "ResUNet-17ch": "resunet_17ch.txt",
    "ResUNet-GAT": "resunet_gat.txt",
}

parsed_logs = {}
for label, filename in log_files.items():
    for base in base_candidates:
        path = base / "assets" / filename
        if path.exists():
            parsed_logs[label] = parse_training_log(path)
            break

if not parsed_logs:
    raise FileNotFoundError("Could not find any of the training log files in the current workspace.")

fig, axes = plt.subplots(1, 2, figsize=(14,5), constrained_layout=True)

for label, data in parsed_logs.items():
    axes[0].plot(data["epochs"], data["train_loss"], label=f"{label} - Train", linewidth=1)
    axes[1].plot(data["epochs"], data["val_loss"], label=f"{label} - Val", linewidth=1)

axes[0].set_title("Training Loss vs. Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_title("Validation Loss vs. Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig("training_and_validation_loss.pdf", format="pdf", dpi=300, bbox_inches="tight", pad_inches=0.02)
plt.savefig("training_and_validation_loss.png", dpi=300, bbox_inches="tight", pad_inches=0.02)
plt.show()

In [ ]:
from pathlib import Path
print(Path.cwd())
print(list(Path.cwd().glob("**/*.txt")))

In [ ]:

import torch
import numpy as np
from torch.utils.data import DataLoader


def evaluate_thresholds(model, dataloader, thresholds=None, device=None):
    if thresholds is None:
        thresholds = np.arange(0.05, 0.96, 0.05)
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    model.eval()

    total_tp = {float(t): 0 for t in thresholds}
    total_fp = {float(t): 0 for t in thresholds}
    total_fn = {float(t): 0 for t in thresholds}

    with torch.no_grad():
        for images, targets in dataloader:
            images = images.to(device)
            targets = targets.to(device)

            logits = model(images)
            probs = torch.softmax(logits, dim=1)[:, 1]

            for t in thresholds:
                pred = (probs > float(t)).long()
                tp = ((targets == 1) & (pred == 1)).sum().item()
                fp = ((targets == 0) & (pred == 1)).sum().item()
                fn = ((targets == 1) & (pred == 0)).sum().item()

                total_tp[float(t)] += tp
                total_fp[float(t)] += fp
                total_fn[float(t)] += fn

    results = []
    for t in thresholds:
        tp = total_tp[float(t)]
        fp = total_fp[float(t)]
        fn = total_fn[float(t)]
        precision = tp / (tp + fp + 1e-9)
        recall = tp / (tp + fn + 1e-9)
        f1 = 2 * precision * recall / (precision + recall + 1e-9)
        iou = tp / (tp + fp + fn + 1e-9)
        results.append((float(t), precision, recall, f1, iou))

    return results

IMG_DIR  = "/content/landlside4sense/validData/img"
MASK_DIR = "/content/landslide4sense/validData/masks"
SAVE_PATH = "/content/landslide4sense_model.pth"

BATCH_SIZE = 16  
EPOCHS     = 75
LEARNING_RATE = 1e-4
GRAPH_TYPE = "local"

all_files = sorted([f for f in os.listdir(IMG_DIR) if f.endswith(".h5")])
all_ids = [int(f.split("_")[1].split(".")[0]) for f in all_files]

# Shuffle with a fixed seed so train and val stays consistent across restarts
random.seed(42)
random.shuffle(all_ids)

train_size = int(0.85 * len(all_ids))
train_ids = all_ids[:train_size]
val_ids = all_ids[train_size:]

print(f"Total Samples: {len(all_ids)} | Train: {len(train_ids)} | Val: {len(val_ids)}")
print("\n--- Computing Normalization Statistics From Training Split ---")
MEANS, STDS = compute_normalization(IMG_DIR, train_ids)


val_dataset = LandslideDataset(img_dir=IMG_DIR, mask_dir=MASK_DIR, transform=val_transform(MEANS, STDS), file_ids=val_ids)

val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

model_path = globals().get("SAVE_PATH") or "/content/drive/MyDrive/best_nepal_model.pth"
print(f"Loading model checkpoint from: {model_path}")

# Ensure the model class is available.
if "ResUNet" not in globals():
    try:
        from src.ResUNet import ResUNet
    except Exception as e:
        raise RuntimeError(
            "Could not import ResUNet. Make sure the notebook path or src package is accessible."
        ) from e

model_for_eval = ResUNet(in_channels=14, num_classes=2)
checkpoint = torch.load(model_path, map_location="cpu")
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model_for_eval.load_state_dict(checkpoint["model_state_dict"])
else:
    model_for_eval.load_state_dict(checkpoint)

thresholds = np.arange(0.05, 0.96, 0.05)
results = evaluate_thresholds(model_for_eval, val_loader, thresholds=thresholds)

best_threshold, best_precision, best_recall, best_f1, best_iou = max(results, key=lambda row: row[3])

print(f"Best threshold by F1 score: {best_threshold:.2f}")
print(f"Precision: {best_precision:.4f}, Recall: {best_recall:.4f}, F1: {best_f1:.4f}, IoU: {best_iou:.4f}\n")
print("Threshold sweep results:")
for t, p, r, f, i in results:
    print(f"{t:.2f}: Precision={p:.4f}, Recall={r:.4f}, F1={f:.4f}, IoU={i:.4f}")

In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
def show_17_bands(h5_path, figsize=(20, 16), cmap='gray', percentile=2):
    """Load a single `.h5` image and display all 17 channels in a clean grid.

    - `h5_path`: path to image_*.h5 file containing dataset field 'img'
    - `percentile`: low/high percentile for contrast stretching to avoid outliers
    """
    if not os.path.exists(h5_path):
        raise FileNotFoundError(h5_path)

    with h5py.File(h5_path, 'r') as f:
        raw = f['img'][:]  # expected shape: (H, W, bands >= 14)

    # Extract channels (mirror the dataset construction used earlier)
    blue  = raw[:, :, 1].astype(np.float32)
    green = raw[:, :, 2].astype(np.float32)
    red   = raw[:, :, 3].astype(np.float32)
    b5    = raw[:, :, 4].astype(np.float32)
    b6    = raw[:, :, 5].astype(np.float32)
    b7    = raw[:, :, 6].astype(np.float32)
    nir   = raw[:, :, 7].astype(np.float32)
    swir1 = raw[:, :, 10].astype(np.float32)
    swir2 = raw[:, :, 11].astype(np.float32)
    slope = raw[:, :, 12].astype(np.float32)
    dem   = raw[:, :, 13].astype(np.float32)

    # Topographic derivatives (function defined earlier in the notebook)
    northness, eastness, curvature = compute_topographical_features(dem, slope)

    eps = 1e-6
    ndvi = (nir - red) / (nir + red + eps)
    bsi = ((swir1 + red) - (nir + blue)) / ((swir1 + red) + (nir + blue) + eps)
    ndwi = (green - nir) / (green + nir + eps)

    bands = [
        dem,
        slope,
        northness,
        eastness,
        curvature,
        blue,
        green,
        red,
        nir,
        b5,
        b6,
        b7,
        swir1,
        swir2,
        ndvi,
        bsi,
        ndwi,
    ]

    names = [
        'DEM', 'Slope', 'Northness', 'Eastness', 'Curvature',
        'Blue', 'Green', 'Red', 'NIR', 'B5', 'B6', 'B7', 'SWIR1', 'SWIR2',
        'NDVI', 'BSI', 'NDWI'
    ]

    # Contrast-stretch each band using percentiles, then clip to [0,1]
    normed = []
    for b in bands:
        lo, hi = np.percentile(b, [percentile, 100 - percentile])
        if hi - lo == 0:
            nb = np.zeros_like(b)
        else:
            nb = np.clip((b - lo) / (hi - lo), 0.0, 1.0)
        normed.append(nb)

    # Layout: 4 rows x 5 cols = 20 slots (last 3 left blank)
    fig, axes = plt.subplots(4, 5, figsize=figsize, constrained_layout=True)
    axes = axes.ravel()

    for i, ax in enumerate(axes):
        ax.axis('off')
        if i < len(normed):
            im = ax.imshow(normed[i], cmap=cmap)
            ax.set_title(f"{i+1}. {names[i]}", fontsize=10)
            # small colorbar per axis for visual reference
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.01)

    plt.suptitle(os.path.basename(h5_path), fontsize=16)
    plt.show()

In [ ]:
from pathlib import Path

img_path = Path("/content/landslide4sense/TrainData/img/image_8.h5")

show_17_bands(str(img_path))



In [ ]:
from pathlib import Path

img_path = Path("/content/Nepal_SR/img/image_222.h5")

print("Checking image path:", img_path)
if not img_path.exists():
    print(f"Image not found: {img_path}")
    print("Check that the dataset is extracted and the path is correct.")
else:
    # Inspect the HDF5 structure to help debug common issues
    try:
        with h5py.File(str(img_path), 'r') as hf:
            keys = list(hf.keys())
            print("HDF5 keys:", keys)
            if 'img' in hf:
                arr = hf['img'][:]
                print("'img' dataset shape:", arr.shape, "dtype:", arr.dtype)
            else:
                print("No 'img' dataset in the file. Available keys:", keys)
    except Exception as e:
        print("Failed to read HDF5 file:", e)

    try:
        # show_17_bands expects a string path
        show_17_bands(str(img_path))
    except NameError as e:
        print("Missing definition:", e)
        print("Make sure `compute_topographical_features` and `show_17_bands` are defined in this notebook or run the cell that defines them.")
    except Exception as e:
        import traceback
        traceback.print_exc()
